# srforge Demo

用合成数据演示完整流程：PySR 多轮训练 → 结构分析 → 公式锻造。

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. 生成合成数据

真实公式：`y = 3.5 * sqrt(x0 * x1) + 2.1 * x2 + x3 + 5.0`

In [2]:
np.random.seed(42)
n = 200
X = pd.DataFrame({
    'x0': np.random.uniform(1, 10, n),
    'x1': np.random.uniform(1, 10, n),
    'x2': np.random.uniform(0, 5, n),
    'x3': np.random.uniform(0, 5, n),
    'noise1': np.random.normal(0, 1, n),   # 噪声特征
    'noise2': np.random.normal(0, 1, n),
})
y = 3.5 * np.sqrt(X['x0'] * X['x1']) + 2.1 * X['x2'] + X['x3'] + 5.0 + np.random.normal(0, 0.5, n)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Train: {len(x_train)}, Test: {len(x_test)}')

Train: 140, Test: 60


## 2. 多轮 PySR 训练

In [3]:
from pysr import PySRRegressor

equations_list = []
for seed in range(10):
    model = PySRRegressor(
        binary_operators=["+", "-", "*", "/", "pow"],
        unary_operators=["sqrt"],
        niterations=200,
        populations=8,
        population_size=30,
        maxsize=15,
        random_state=seed,
        deterministic=True,
        parallelism="serial",
        verbosity=0,
    )
    model.fit(x_train, y_train)
    equations_list.append(model.equations_)

print(f'训练完成，{len(equations_list)} 轮')

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


d:\py_projects\srforge\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
d:\py_projects\srforge\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(
d:\py_projects\srforge\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
d:\py_projects\srforge\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One

训练完成，10 轮


## 3. 一键分析

In [4]:
from srforge import build_formulas, full_analysis

formulas = build_formulas(equations_list)
result = full_analysis(formulas)
# 终端打印 Pattern 报告 + 公式排名 + 浏览器打开 HTML 报告


Pattern #1
Signature : Pow(VAR,CONST)
Frequency : 62

Coverage
  Run     : 10 / 10 (100.0%)
  Formula : 61 / 112 (54.5%)

Score
  Total      : 0.740   综合评分
  Run        : 1.000   越接近1说明跨Run重复出现
  Formula    : 0.545   越接近1说明覆盖更多公式
  Stability  : 0.782   越接近1说明变量位置越稳定
  Frequency  : 0.204   出现频率(归一化)

Evidence
  Strong

Slots

  arg_0
   Distribution:
     x0     56.5%
     x1     37.1%
     x3     3.2%
     x2     3.2%
   Confidence: 0.409
   Top1: x0 (0.56)
   Top2 ratio: 0.37

  arg_1
   Distribution:
     <const> 100.0%
   Confidence: 1.000
   Top1: <const> (1.00)
   Top2 ratio: 0.00

Recommendation
  建议保留

Pattern #2
Signature : Add(VAR,VAR)
Frequency : 37

Coverage
  Run     : 10 / 10 (100.0%)
  Formula : 37 / 112 (33.0%)

Score
  Total      : 0.682   综合评分
  Run        : 1.000   越接近1说明跨Run重复出现
  Formula    : 0.330   越接近1说明覆盖更多公式
  Stability  : 0.851   越接近1说明变量位置越稳定
  Frequency  : 0.122   出现频率(归一化)

Evidence
  Strong

Slots

  arg_0
   Distribution:
     x1     70.3%
     x0     29

## 4. 提取共识子式

In [5]:
from srforge import extract_candidates, print_candidates

candidates = extract_candidates(formulas, result["scored"],
                                 result["patterns"], result["context"],
                                 x_filter=x_train)
print_candidates(candidates, topk=10)


#    Subexpr                                       Pat     Freq   nFeat 
1    x1 + x2                                       0.681   37     2     
2    x0 + x2 + 2.7658658                           0.627   20     2     
3    x2*sqrt(x3)                                   0.615   11     2     
4    sqrt(x0)*(x1 + x2)                            0.448   16     3     
5    x3 + (sqrt(x0) - 0.39803854)*(x1 + x2) +...   0.415   5      4     
6    (sqrt(x0) - 0.35285485)*(x1 + x2) + 15.3...   0.415   5      3     
7    x0*x1                                         0.390   10     2     
8    sqrt(x0*x1)                                   0.387   9      2     
9    (sqrt(x0) - 0.39803854)*(x1 + x2)             0.350   10     3     
10   3.48202570352374*sqrt(x0*x1)                  0.341   7      2     


## 5. ElasticNet 锻造公式

In [6]:
from srforge import elastic_select, print_selection

sel = elastic_select(candidates, x_train, y_train, x_test, y_test)
print_selection(sel)
# 输出最终公式 + 测试集 R²/MSE


Elastic Net 筛选结果
截距: 12.516519

选中 5 项:
  +0.262454  ×  x1 + x2
  +0.320158  ×  x0 + x2 + 2.7658658
  +0.560770  ×  x2*sqrt(x3)
  +0.304433  ×  sqrt(x0)*(x1 + x2)
  +0.165334  ×  x0*x1

最终公式:
  12.516519 + 0.262454*(x1 + x2) + 0.320158*(x0 + x2 + 2.7658658) + 0.560770*x2*sqrt(x3) + 0.304433*(sqrt(x0)*(x1 + x2)) + 0.165334*x0*x1

测试集: MSE=1.9831  R²=0.9639


## 6. 稳定性验证

In [7]:
from srforge import cross_validate, print_cv, check_formula

df = cross_validate(sel["formula"], X, y)
print_cv(df)

# 安全检查
warnings = check_formula(sel["formula"], X)
if warnings:
    for w in warnings:
        print(f"风险:{w}")
else:
    print("公式安全")

seed     R²        MSE        y_mean     y_std      bad   
-------------------------------------------------------
0.0      0.9733    1.53       30.75      7.57       13.0   ★
5.0      0.9755    1.33       31.05      7.38       11.0   ★
10.0     0.9701    1.50       30.68      7.09       10.0   ★
15.0     0.9801    1.10       30.93      7.44       16.0   ★
20.0     0.9729    1.64       30.52      7.80       10.0   ★
25.0     0.9750    1.21       30.48      6.95       8.0    ★
30.0     0.9662    1.85       30.80      7.41       12.0   ★
35.0     0.9750    1.47       31.03      7.67       20.0   ★
40.0     0.9698    1.51       30.86      7.07       13.0   ★
45.0     0.9688    1.69       30.92      7.35       17.0   ★
50.0     0.9769    1.42       30.71      7.83       13.0   ★

均值 R²=0.9731  最佳=0.9801  最差=0.9662
公式安全


## 7. PySR Round 2（二次搜索）

可选：将共识子式喂回第二轮搜索

In [ ]:
# from srforge import round2_forge, expand_feats
# 
# # 需要定义 train_fn 函数
# r2 = round2_forge(result, formulas, x_train, y_train, x_test, y_test, train_fn=your_train_fn)
# 
# # 展开 feat_N → 原始表达式
# top_eq = r2["scored"][0]["equation"]
# eq = expand_feats(top_eq, r2["feat_map"])
# print(eq)